<a href="https://colab.research.google.com/github/Harshit-077/Non-Manual-Features-in-Indian-Sign-Language/blob/main/isl_nmf_detection_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ISL Non-Manual Feature (NMF) Detection
### Facial expressions · Head movement · Eye/Brow features in Indian Sign Language

**Datasets:**
| Dataset | Kaggle URL | Purpose |
|---|---|---|
| AffectNet | `mstjebashazida/affectnet` | Expression pretraining (8 classes, ~440K images) |
| ISL-CSLTR | `drblack00/isl-csltr-indian-sign-language-dataset` | ISL videos for NMF fine-tuning (700 videos, 100 sentences, 7 signers) |

**Pipeline:**
1. Download both datasets via Kaggle API  
2. Explore & visualise data  
3. Extract MediaPipe landmarks from ISL-CSLTR video frames  
4. Train three NMF streams (Expression CNN · Head Movement TCN · Eye/Brow MLP)  
5. Late-fusion classifier  
6. Evaluate + ablation  
7. ONNX export for real-time deployment

## 1. Environment Setup

In [1]:
# Install all required packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q mediapipe opencv-python-headless albumentations timm
!pip install -q pandas numpy scikit-learn matplotlib seaborn tqdm kaggle

import os, cv2, json, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict, Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from torchvision import datasets as tv_datasets

import mediapipe as mp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import timm

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 14.7 MB/s eta 0:00:00
Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


## 2. Kaggle Dataset Download

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Set up Kaggle credentials
# ─────────────────────────────────────────────────────────────────────────────
# Option A – kaggle.json (recommended)
#   1. Go to https://www.kaggle.com/settings → API → Create New Token
#   2. A kaggle.json file downloads automatically. Run the cell below to upload it.
#
# Option B – environment variables (for Colab secrets / CI)
#   Set KAGGLE_USERNAME and KAGGLE_KEY before running.

# ── Upload kaggle.json in Jupyter ──
# from google.colab import files   # uncomment if on Colab
# files.upload()                   # then select kaggle.json

# ── Or paste credentials directly (never commit to git!) ──
# import os
# os.environ['KAGGLE_USERNAME'] = 'YOUR_USERNAME'
# os.environ['KAGGLE_KEY']      = 'YOUR_API_KEY'

# ── Move kaggle.json to the expected location ──
#!mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json   # uncomment after uploading
#!chmod 600 ~/.kaggle/kaggle.json 2>/dev/null || echo 'kaggle.json not yet placed — see instructions above'

# Directory layout
BASE        = Path('isl_nmf')
DATA        = BASE / 'data'
AFFECTNET   = DATA / 'affectnet'
ISL_DIR     = DATA / 'isl_csltr'
FRAMES_DIR  = DATA / 'isl_frames'   # extracted video frames
LM_DIR      = DATA / 'landmarks'    # saved .npz landmark files
CKPT        = BASE / 'checkpoints'
LOGS        = BASE / 'logs'

for d in [AFFECTNET, ISL_DIR, FRAMES_DIR, LM_DIR, CKPT, LOGS]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories created.')

Directories created.


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Download datasets  (run once — skip if already downloaded)
# ─────────────────────────────────────────────────────────────────────────────

# ── AffectNet ──────────────────────────────────────────────────────────────
# Dataset: mstjebashazida/affectnet
# Structure after unzip:
#   affectnet/
#     train/
#       0_neutral/   1_happy/   2_sad/   3_surprise/
#       4_fear/      5_disgust/ 6_anger/ 7_contempt/
#     val/
#       (same 8 subfolders)
#
# ~440K training images, 4K validation images
# Class IDs: 0=neutral, 1=happy, 2=sad, 3=surprise, 4=fear, 5=disgust, 6=anger, 7=contempt

if not any(AFFECTNET.iterdir()) if AFFECTNET.exists() else True:
    print('Downloading AffectNet...')
    !kaggle datasets download -d mstjebashazida/affectnet \
        -p {AFFECTNET} --unzip --quiet
    print('AffectNet download complete.')
else:
    print('AffectNet already present — skipping download.')

# ── ISL-CSLTR ──────────────────────────────────────────────────────────────
# Dataset: drblack00/isl-csltr-indian-sign-language-dataset
# Structure after unzip:
#   isl_csltr/
#     Videos/          <- 700 .mp4 / .avi files, one per sentence
#     Frames/          <- 18,863 pre-extracted sentence-level frames
#     WordImages/      <- 1,036 word-level images (100 words)
#     Annotations/     <- CSV / JSON with sentence IDs, signer IDs, time boundaries
#
# The dataset has 100 spoken-language sentences, 7 signers

if not any(ISL_DIR.iterdir()) if ISL_DIR.exists() else True:
    print('Downloading ISL-CSLTR...')
    !kaggle datasets download -d drblack00/isl-csltr-indian-sign-language-dataset \
        -p {ISL_DIR} --unzip --quiet
    print('ISL-CSLTR download complete.')
else:
    print('ISL-CSLTR already present — skipping download.')

Dataset URL: https://www.kaggle.com/datasets/mstjebashazida/affectnet
License(s): MIT
AffectNet download complete.
Dataset URL: https://www.kaggle.com/datasets/drblack00/isl-csltr-indian-sign-language-dataset
License(s): CC-BY-SA-4.0
ISL-CSLTR download complete.


## 3. Dataset Exploration

In [5]:
# ── AffectNet class mapping ────────────────────────────────────────────────
AFFECTNET_CLASSES = {
    0: 'neutral',   1: 'happy',    2: 'sad',      3: 'surprise',
    4: 'fear',      5: 'disgust',  6: 'anger',    7: 'contempt'
}

# ISL NMF mapping — which expression maps to which grammatical NMF
EXPR_TO_NMF = {
    'neutral':  'neutral',        # baseline
    'happy':    'positive',       # affirmative markers
    'sad':      'furrowed_brows', # negation facial
    'surprise': 'raised_brows',   # wh-questions
    'fear':     'raised_brows',   # wh-question secondary
    'disgust':  'furrowed_brows', # negation
    'anger':    'furrowed_brows', # negation / intensity
    'contempt': 'furrowed_brows'  # negation
}

def count_affectnet_split(split: str):
    split_dir = AFFECTNET / split
    if not split_dir.exists():
        return None
    counts = {}
    for cls_folder in sorted(split_dir.iterdir()):
        if cls_folder.is_dir():
            n = len(list(cls_folder.glob('*.jpg')) + list(cls_folder.glob('*.png')))
            counts[cls_folder.name] = n
    return counts

train_counts = count_affectnet_split('train')
val_counts   = count_affectnet_split('val')

if train_counts:
    df_counts = pd.DataFrame({'Train': train_counts, 'Val': val_counts or {}}).fillna(0).astype(int)
    print('AffectNet class distribution:')
    print(df_counts)
    print(f'Total train: {df_counts["Train"].sum():,}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    df_counts['Train'].plot(kind='bar', ax=axes[0], color='#3B8BD4', edgecolor='none')
    axes[0].set_title('AffectNet — Train set distribution', fontsize=12)
    axes[0].set_ylabel('Image count')
    axes[0].tick_params(axis='x', rotation=30)

    if val_counts:
        df_counts['Val'].plot(kind='bar', ax=axes[1], color='#E85D24', edgecolor='none')
        axes[1].set_title('AffectNet — Val set distribution', fontsize=12)
        axes[1].tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.savefig(LOGS / 'affectnet_distribution.png', dpi=120)
    plt.show()
else:
    print('[Waiting for download] AffectNet not found at', AFFECTNET)
    print('Run Section 2 cells first.')

[Waiting for download] AffectNet not found at isl_nmf/data/affectnet
Run Section 2 cells first.


In [ ]:
def show_affectnet_samples(n_per_class: int = 3):
    """Display sample images from each AffectNet class."""
    train_dir = AFFECTNET / 'train'
    if not train_dir.exists():
        print('AffectNet train dir not found.'); return

    class_dirs = sorted(train_dir.iterdir())
    n_classes = len(class_dirs)
    fig, axes = plt.subplots(n_classes, n_per_class,
                              figsize=(n_per_class * 2.5, n_classes * 2.5))

    for row, cls_dir in enumerate(class_dirs):
        imgs = list(cls_dir.glob('*.jpg'))[:n_per_class]
        for col in range(n_per_class):
            ax = axes[row][col] if n_classes > 1 else axes[col]
            if col < len(imgs):
                img = cv2.cvtColor(cv2.imread(str(imgs[col])), cv2.COLOR_BGR2RGB)
                ax.imshow(img)
                if col == 0:
                    ax.set_ylabel(cls_dir.name, fontsize=9, rotation=0,
                                   labelpad=60, va='center')
            ax.axis('off')

    plt.suptitle('AffectNet — Sample images per class', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(LOGS / 'affectnet_samples.png', dpi=100, bbox_inches='tight')
    plt.show()

show_affectnet_samples(n_per_class=4)

In [ ]:
# ── ISL-CSLTR Exploration ─────────────────────────────────────────────────
def explore_isl_dataset(isl_dir: Path):
    print('=== ISL-CSLTR Dataset Overview ===')
    print(f'Root: {isl_dir}\n')

    # List top-level contents
    for item in sorted(isl_dir.iterdir()):
        if item.is_dir():
            children = list(item.iterdir())
            print(f'  [{item.name}/]  — {len(children)} items')
            for child in sorted(children)[:5]:
                print(f'      {child.name}')
            if len(children) > 5:
                print(f'      ... ({len(children)-5} more)')
        else:
            print(f'  {item.name}  ({item.stat().st_size / 1e6:.1f} MB)')

    # Count videos
    video_exts = ['*.mp4', '*.avi', '*.mov', '*.mkv']
    videos = []
    for ext in video_exts:
        videos.extend(isl_dir.rglob(ext))
    print(f'\nTotal video files found: {len(videos)}')

    # Count frames
    frame_exts = ['*.jpg', '*.png', '*.jpeg']
    frames = []
    for ext in frame_exts:
        frames.extend(isl_dir.rglob(ext))
    print(f'Total image files found: {len(frames)}')

    # Count annotation files
    ann_files = list(isl_dir.rglob('*.csv')) + list(isl_dir.rglob('*.json'))
    print(f'Annotation files found: {len(ann_files)}')
    for f in ann_files:
        print(f'  {f.relative_to(isl_dir)}')

    return videos, frames, ann_files

if ISL_DIR.exists() and any(ISL_DIR.iterdir()):
    isl_videos, isl_frames, isl_anns = explore_isl_dataset(ISL_DIR)
else:
    print('[Waiting] ISL-CSLTR not downloaded yet.')
    isl_videos, isl_frames, isl_anns = [], [], []

In [ ]:
def load_isl_annotations(isl_dir: Path):
    """
    Load ISL-CSLTR annotations.
    Tries CSV first, then JSON. Returns a unified DataFrame.

    Expected columns in annotation files:
      video_id / file_name, sentence_id, sentence_text, signer_id, start_frame, end_frame
    """
    ann_csvs = list(isl_dir.rglob('*.csv'))
    ann_jsons = list(isl_dir.rglob('*.json'))

    dfs = []
    for csv_path in ann_csvs:
        try:
            df = pd.read_csv(csv_path)
            df['source_file'] = csv_path.name
            dfs.append(df)
            print(f'Loaded CSV: {csv_path.name}  shape={df.shape}')
            print(f'  Columns: {list(df.columns)}')
            print(df.head(3).to_string())
            print()
        except Exception as e:
            print(f'Failed to read {csv_path.name}: {e}')

    for json_path in ann_jsons:
        try:
            with open(json_path) as f:
                data = json.load(f)
            if isinstance(data, list):
                df = pd.DataFrame(data)
            elif isinstance(data, dict):
                df = pd.DataFrame(list(data.values()))
            df['source_file'] = json_path.name
            dfs.append(df)
            print(f'Loaded JSON: {json_path.name}  shape={df.shape}')
            print(f'  Columns: {list(df.columns)}')
            print()
        except Exception as e:
            print(f'Failed to read {json_path.name}: {e}')

    if not dfs:
        print('No annotation files loaded. Using filename-based inference.')
        return None

    return pd.concat(dfs, ignore_index=True)

df_isl_ann = load_isl_annotations(ISL_DIR)

## 4. Frame Extraction from ISL-CSLTR Videos

In [ ]:
def extract_frames_from_video(video_path: Path, out_dir: Path,
                               target_fps: int = 10,
                               max_frames: int = None) -> list:
    """
    Extract frames from a video at target_fps.
    Saves as: out_dir/{video_stem}_frame_{N:05d}.jpg
    Returns list of saved frame paths.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frame_interval = max(1, int(round(src_fps / target_fps)))

    saved, frame_idx = [], 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            save_path = out_dir / f'{video_path.stem}_f{frame_idx:06d}.jpg'
            cv2.imwrite(str(save_path), frame)
            saved.append(save_path)
            if max_frames and len(saved) >= max_frames:
                break
        frame_idx += 1

    cap.release()
    return saved


def batch_extract_isl_frames(isl_dir: Path, out_dir: Path,
                              target_fps: int = 10,
                              max_videos: int = None) -> pd.DataFrame:
    """
    Extract frames from all ISL-CSLTR videos.
    Returns DataFrame: [video_path, frame_path, video_id, signer_id]
    """
    video_exts = ['.mp4', '.avi', '.mov', '.mkv']
    videos = [p for p in isl_dir.rglob('*') if p.suffix.lower() in video_exts]

    if max_videos:
        videos = videos[:max_videos]

    records = []
    for vid_path in tqdm(videos, desc='Extracting frames'):
        vid_out = out_dir / vid_path.stem
        if vid_out.exists() and any(vid_out.iterdir()):
            frames = list(vid_out.glob('*.jpg'))
        else:
            frames = extract_frames_from_video(vid_path, vid_out, target_fps)

        # Parse signer info from filename if encoded (e.g. S1_sentence01.mp4)
        stem = vid_path.stem
        signer_id  = stem.split('_')[0] if '_' in stem else 'unknown'
        sentence_id = stem.split('_')[1] if len(stem.split('_')) > 1 else stem

        for fp in frames:
            records.append({
                'video_path': str(vid_path),
                'frame_path': str(fp),
                'video_id': stem,
                'signer_id': signer_id,
                'sentence_id': sentence_id,
            })

    df = pd.DataFrame(records)
    print(f'Total frames extracted: {len(df):,}')
    print(f'Unique videos: {df["video_id"].nunique()}')
    print(f'Unique signers: {df["signer_id"].nunique()}')
    return df


# Run extraction (set max_videos=None to process all 700 videos)
# Recommend: max_videos=50 for quick development, None for full training
if isl_videos:
    df_isl_frames = batch_extract_isl_frames(ISL_DIR, FRAMES_DIR,
                                              target_fps=10, max_videos=50)
    df_isl_frames.to_csv(DATA / 'isl_frames_manifest.csv', index=False)
    print('Manifest saved.')
else:
    print('No ISL videos found. Download dataset first.')
    df_isl_frames = pd.DataFrame()

In [ ]:
def show_isl_frames(df: pd.DataFrame, n: int = 12):
    """Show a grid of randomly sampled ISL-CSLTR frames."""
    if df.empty:
        print('No frames to display.'); return
    sample = df.sample(min(n, len(df)), random_state=SEED)
    cols = 4
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()
    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = cv2.cvtColor(cv2.imread(row['frame_path']), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{row['signer_id']} | {row['sentence_id']}",
                     fontsize=7, pad=2)
        ax.axis('off')
    for ax in axes[len(sample):]:
        ax.axis('off')
    plt.suptitle('ISL-CSLTR — Sampled video frames', fontsize=12)
    plt.tight_layout()
    plt.savefig(LOGS / 'isl_frames_sample.png', dpi=100)
    plt.show()

show_isl_frames(df_isl_frames)

## 5. MediaPipe Landmark Extraction

In [ ]:
mp_face = mp.solutions.face_mesh
mp_draw = mp.solutions.drawing_utils

# Key landmark index groups
LM_GROUPS = {
    'left_brow':   [70, 63, 105, 66, 107],
    'right_brow':  [336, 296, 334, 293, 300],
    'left_eye':    [33, 160, 158, 133, 153, 144],
    'right_eye':   [362, 385, 387, 263, 373, 380],
    'mouth':       [61, 291, 0, 17, 402, 182],
    'nose_tip':    [1],
    'chin':        [199],
    'left_ear':    [234],
    'right_ear':   [454],
    'forehead':    [10],
    'inner_brows': [107, 336],   # for furrow distance
}

def eye_aspect_ratio(pts: np.ndarray) -> float:
    """Standard EAR from 6 eye landmarks."""
    A = np.linalg.norm(pts[1] - pts[5])
    B = np.linalg.norm(pts[2] - pts[4])
    C = np.linalg.norm(pts[0] - pts[3])
    return (A + B) / (2.0 * C + 1e-6)


def extract_features(frame_bgr: np.ndarray, face_mesh) -> dict | None:
    """
    Extract all NMF features from one BGR frame.
    Returns dict or None if no face detected.
    """
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res = face_mesh.process(rgb)
    if not res.multi_face_landmarks:
        return None

    lm = res.multi_face_landmarks[0].landmark
    pts = np.array([[l.x, l.y, l.z] for l in lm], dtype=np.float32)  # (478, 3)

    # Head pose (simplified from facial geometry)
    nose, chin   = pts[1], pts[199]
    l_ear, r_ear = pts[234], pts[454]
    forehead     = pts[10]

    pitch = float(np.degrees(np.arctan2(chin[1] - forehead[1],
                                         abs(chin[2] - forehead[2]) + 1e-6)))
    yaw   = float(np.degrees(np.arctan2(r_ear[0] - l_ear[0],
                                         abs(r_ear[2] - l_ear[2]) + 1e-6)))
    roll  = float(np.degrees(np.arctan2(r_ear[1] - l_ear[1],
                                         abs(r_ear[0] - l_ear[0]) + 1e-6)))

    # Eye features
    l_eye_pts = pts[LM_GROUPS['left_eye']]
    r_eye_pts = pts[LM_GROUPS['right_eye']]
    ear_l = eye_aspect_ratio(l_eye_pts)
    ear_r = eye_aspect_ratio(r_eye_pts)

    # Brow features
    l_brow_y  = pts[LM_GROUPS['left_brow']][:, 1].mean()
    r_brow_y  = pts[LM_GROUPS['right_brow']][:, 1].mean()
    l_eye_y   = l_eye_pts[:, 1].mean()
    r_eye_y   = r_eye_pts[:, 1].mean()
    brow_raise_l = float(l_eye_y - l_brow_y)   # +ve = raised
    brow_raise_r = float(r_eye_y - r_brow_y)
    furrow_dist  = float(abs(pts[336][0] - pts[107][0]))  # inner brow gap

    # Face bounding box in pixel coords (for face crop)
    h, w = frame_bgr.shape[:2]
    x1 = max(0, int(pts[:, 0].min() * w) - 15)
    y1 = max(0, int(pts[:, 1].min() * h) - 15)
    x2 = min(w, int(pts[:, 0].max() * w) + 15)
    y2 = min(h, int(pts[:, 1].max() * h) + 15)

    return {
        'landmarks':     pts,
        'head_pose':     np.array([pitch, yaw, roll], dtype=np.float32),
        'eye_features':  np.array([ear_l, ear_r, (ear_l+ear_r)/2], dtype=np.float32),
        'brow_features': np.array([brow_raise_l, brow_raise_r,
                                    (brow_raise_l+brow_raise_r)/2,
                                    furrow_dist], dtype=np.float32),
        'face_bbox':     (x1, y1, x2, y2),
    }


print('Feature extraction function ready.')
print('EAR: Eye Aspect Ratio (higher = more open)')
print('Brow raise: eye_y - brow_y (higher = brow further above eye = raised)')

In [ ]:
def batch_extract_and_save(df_frames: pd.DataFrame, lm_dir: Path,
                            max_frames: int = None) -> pd.DataFrame:
    """
    Run landmark extraction on all ISL frames and save .npz feature files.
    Returns updated DataFrame with 'lm_path' column.
    """
    lm_dir.mkdir(exist_ok=True)
    subset = df_frames.head(max_frames) if max_frames else df_frames
    records, failed = [], 0

    with mp_face.FaceMesh(
        static_image_mode=True, max_num_faces=1,
        refine_landmarks=True, min_detection_confidence=0.5
    ) as fm:
        for i, row in tqdm(subset.iterrows(), total=len(subset),
                           desc='Extracting landmarks'):
            out_path = lm_dir / f'{i:07d}.npz'
            if out_path.exists():
                records.append({**row.to_dict(), 'lm_path': str(out_path)})
                continue

            img = cv2.imread(str(row['frame_path']))
            if img is None:
                failed += 1; continue

            feat = extract_features(img, fm)
            if feat is None:
                failed += 1; continue

            np.savez_compressed(
                out_path,
                landmarks=feat['landmarks'],
                head_pose=feat['head_pose'],
                eye_features=feat['eye_features'],
                brow_features=feat['brow_features'],
                face_bbox=np.array(feat['face_bbox'])
            )
            records.append({**row.to_dict(), 'lm_path': str(out_path)})

    print(f'Saved: {len(records)} | Skipped (no face): {failed}')
    return pd.DataFrame(records)


if not df_isl_frames.empty:
    df_isl_lm = batch_extract_and_save(df_isl_frames, LM_DIR, max_frames=5000)
    df_isl_lm.to_csv(DATA / 'isl_landmarks.csv', index=False)
else:
    df_isl_lm = pd.DataFrame()
    print('Skipping landmark extraction — no frames available.')

In [ ]:
def visualise_nmf_on_frame(frame_path: str, face_mesh):
    """Overlay MediaPipe landmarks and NMF values on a frame."""
    img = cv2.imread(str(frame_path))
    if img is None:
        print('Image not found'); return

    feat = extract_features(img, face_mesh)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: raw frame
    axes[0].imshow(rgb)
    axes[0].set_title('Original frame', fontsize=11)
    axes[0].axis('off')

    # Right: landmarks + NMF annotations
    if feat:
        res = face_mesh.process(rgb)
        overlay = rgb.copy()
        if res.multi_face_landmarks:
            mp_draw.draw_landmarks(
                overlay, res.multi_face_landmarks[0],
                mp_face.FACEMESH_CONTOURS,
                mp_draw.DrawingSpec(color=(0, 200, 80), thickness=1, circle_radius=1),
                mp_draw.DrawingSpec(color=(50, 150, 255), thickness=1)
            )
        axes[1].imshow(overlay)
        axes[1].set_title(
            f'NMF Features\n'
            f'Pitch: {feat["head_pose"][0]:.1f}°  '
            f'Yaw: {feat["head_pose"][1]:.1f}°  '
            f'Roll: {feat["head_pose"][2]:.1f}°\n'
            f'EAR: {feat["eye_features"][2]:.3f}  '
            f'Brow raise: {feat["brow_features"][2]:.3f}  '
            f'Furrow: {feat["brow_features"][3]:.3f}',
            fontsize=9
        )
    else:
        axes[1].text(0.5, 0.5, 'No face detected', ha='center', va='center')
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig(LOGS / 'landmark_demo.png', dpi=120)
    plt.show()


if not df_isl_frames.empty:
    sample_frame = df_isl_frames['frame_path'].iloc[0]
    with mp_face.FaceMesh(static_image_mode=True, refine_landmarks=True,
                           min_detection_confidence=0.5) as fm:
        visualise_nmf_on_frame(sample_frame, fm)

## 6. Build Head Movement Sequences from ISL Frames

In [ ]:
SEQ_LEN = 30   # frames per sequence

# Head movement classes for ISL grammar
HEAD_CLASSES = {0: 'nod', 1: 'shake', 2: 'tilt', 3: 'forward', 4: 'still'}

def load_pose_sequence(df_lm: pd.DataFrame, video_id: str) -> np.ndarray | None:
    """
    Load head pose (pitch, yaw, roll) sequence for all frames of one video.
    Returns (T, 3) array or None.
    """
    rows = df_lm[df_lm['video_id'] == video_id].sort_values('frame_path')
    if len(rows) < 5:
        return None

    poses = []
    for _, r in rows.iterrows():
        npz = np.load(r['lm_path'])
        poses.append(npz['head_pose'])

    return np.stack(poses, axis=0).astype(np.float32)  # (T, 3)


def sliding_window_sequences(pose_arr: np.ndarray, seq_len: int,
                              stride: int = 15) -> list:
    """Split a pose time series into overlapping windows of length seq_len."""
    seqs = []
    T = len(pose_arr)
    for start in range(0, T - seq_len + 1, stride):
        window = pose_arr[start: start + seq_len]  # (seq_len, 3)
        # Normalize within window
        mu, sigma = window.mean(0), window.std(0) + 1e-6
        window = (window - mu) / sigma
        seqs.append(window)
    return seqs


def heuristic_head_label(seq: np.ndarray) -> int:
    """
    Assign a head movement label based on dominant axis of motion.
    pitch dominant → nod, yaw dominant → shake, roll dominant → tilt,
    all low → still, pitch positive mean → forward.
    """
    std_p, std_y, std_r = seq[:, 0].std(), seq[:, 1].std(), seq[:, 2].std()
    mean_p = seq[:, 0].mean()

    total_motion = std_p + std_y + std_r
    if total_motion < 0.3:
        return 4  # still

    dominant = np.argmax([std_p, std_y, std_r])
    if dominant == 0:
        return 3 if mean_p > 0.4 else 0  # forward vs nod
    elif dominant == 1:
        return 1  # shake
    else:
        return 2  # tilt


def build_head_movement_dataset(df_lm: pd.DataFrame,
                                  seq_len: int = SEQ_LEN,
                                  stride: int = 10,
                                  augment_synthetic: bool = True):
    """
    Build head movement sequences from ISL-CSLTR landmarks.
    Augments with synthetic sequences if real data is sparse.
    """
    X, y = [], []

    if not df_lm.empty and 'video_id' in df_lm.columns:
        for vid_id in tqdm(df_lm['video_id'].unique(), desc='Building sequences'):
            pose = load_pose_sequence(df_lm, vid_id)
            if pose is None:
                continue
            windows = sliding_window_sequences(pose, seq_len, stride)
            for w in windows:
                label = heuristic_head_label(w)
                X.append(w)
                y.append(label)

        print(f'Real ISL sequences: {len(X)}')
        label_counts = Counter(y)
        for c, n in sorted(label_counts.items()):
            print(f'  {HEAD_CLASSES[c]}: {n}')

    # Augment with synthetic data to balance classes
    if augment_synthetic:
        n_synth_per_class = max(500, 500 - min(Counter(y).values() if y else [0]))
        print(f'\nAdding {n_synth_per_class} synthetic sequences per class...')

        synth_params = {
            'nod':     {'axis': 0, 'amp': (8, 18),  'freq': (1.5, 2.5)},
            'shake':   {'axis': 1, 'amp': (8, 20),  'freq': (1.5, 2.5)},
            'tilt':    {'axis': 2, 'amp': (5, 15),  'freq': (0.5, 1.0)},
            'forward': {'axis': 0, 'amp': (10, 20), 'freq': (0.3, 0.6)},
            'still':   {'axis': -1, 'amp': (0, 0),  'freq': (0, 0)},
        }
        t = np.linspace(0, 2 * np.pi, seq_len)

        for class_id, class_name in HEAD_CLASSES.items():
            p = synth_params[class_name]
            for _ in range(n_synth_per_class):
                seq = np.zeros((seq_len, 3), dtype=np.float32)
                if p['axis'] >= 0 and p['amp'][1] > 0:
                    amp  = np.random.uniform(*p['amp'])
                    freq = np.random.uniform(*p['freq'])
                    seq[:, p['axis']] = amp * np.sin(freq * t)
                seq += np.random.randn(seq_len, 3).astype(np.float32) * 1.5
                # Normalize
                mu, sigma = seq.mean(0), seq.std(0) + 1e-6
                X.append((seq - mu) / sigma)
                y.append(class_id)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    perm = np.random.permutation(len(X))
    print(f'\nFinal dataset: {len(X)} sequences, {len(HEAD_CLASSES)} classes')
    return X[perm], y[perm]


X_hm, y_hm = build_head_movement_dataset(df_isl_lm, seq_len=SEQ_LEN)
np.savez_compressed(DATA / 'head_movement_seqs.npz', X=X_hm, y=y_hm)

## 7. Dataset Classes & DataLoaders

In [ ]:
# ─── AffectNet Expression Dataset ────────────────────────────────────────────
# Uses torchvision ImageFolder since AffectNet Kaggle mirror has class subfolders

AFFECTNET_CLASS_NAMES = ['neutral','happy','sad','surprise','fear','disgust','anger','contempt']
NUM_EXPR_CLASSES = 8  # use 7 if you want to drop 'contempt' (rarer class)

train_tf = A.Compose([
    A.Resize(112, 112),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(0.25, 0.25, p=0.6),
    A.Rotate(limit=20, border_mode=cv2.BORDER_REFLECT, p=0.5),
    A.OneOf([
        A.GaussNoise(var_limit=(5, 30)),
        A.GaussianBlur(blur_limit=(3, 5)),
        A.ImageCompression(quality_lower=60),
    ], p=0.4),
    A.CoarseDropout(max_holes=4, max_height=14, max_width=14, p=0.25),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(112, 112),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])


class AffectNetDataset(Dataset):
    """
    Wraps AffectNet Kaggle mirror (ImageFolder-style).
    affectnet/train/{class_name}/*.jpg
    affectnet/val/{class_name}/*.jpg
    Falls back to synthetic random tensors if not downloaded.
    """
    def __init__(self, root: Path, split: str = 'train',
                  transform=None, max_samples: int = None):
        self.transform = transform
        self.split_dir = root / split
        self.synthetic = not self.split_dir.exists()

        if self.synthetic:
            n = max_samples or 5000
            self.labels = np.random.randint(0, NUM_EXPR_CLASSES, n)
            self.paths  = [None] * n
            print(f'[SYNTHETIC] {split}: {n} random samples (AffectNet not found)')
        else:
            self.paths, self.labels = [], []
            for cls_id, cls_name in enumerate(AFFECTNET_CLASS_NAMES):
                cls_dir = self.split_dir / f'{cls_id}_{cls_name}'
                if not cls_dir.exists():
                    # Try plain class name
                    cls_dir = self.split_dir / cls_name
                if not cls_dir.exists():
                    continue
                imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
                if max_samples:
                    imgs = imgs[:max_samples // NUM_EXPR_CLASSES]
                self.paths.extend(imgs)
                self.labels.extend([cls_id] * len(imgs))

            print(f'AffectNet {split}: {len(self.paths):,} images')
            label_dist = Counter(self.labels)
            for cls_id, cnt in sorted(label_dist.items()):
                print(f'  {AFFECTNET_CLASS_NAMES[cls_id]:<12}: {cnt:,}')

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        label = int(self.labels[idx])
        if self.synthetic or self.paths[idx] is None:
            return torch.randn(3, 112, 112), label
        img = cv2.imread(str(self.paths[idx]))
        if img is None:
            return torch.zeros(3, 112, 112), label
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label


# ─── Head Movement Dataset ────────────────────────────────────────────────────
class HeadMovementDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, augment: bool = False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            x += torch.randn_like(x) * 0.05
            x *= 0.9 + torch.rand(1) * 0.2
        return x, self.y[idx]


# ─── Eye/Brow Dataset (from real ISL landmarks) ───────────────────────────────
class EyeBrowDataset(Dataset):
    """
    Builds eye/brow feature vectors from saved .npz landmark files.
    Labels are heuristically assigned based on EAR and brow displacement.
    Classes: 0=neutral, 1=raised_brows (wh-Q), 2=furrowed_brows (negation)
    """
    CLASS_NAMES = {0: 'neutral', 1: 'raised_brows', 2: 'furrowed_brows'}

    def __init__(self, df_lm: pd.DataFrame = None, n_synthetic: int = 4000):
        self.X, self.y = self._load(df_lm, n_synthetic)

    def _load(self, df_lm, n_synth):
        X, y = [], []

        if df_lm is not None and not df_lm.empty and 'lm_path' in df_lm.columns:
            for _, row in df_lm.iterrows():
                try:
                    npz = np.load(row['lm_path'])
                    eye = npz['eye_features']    # [ear_l, ear_r, ear_avg]
                    brow = npz['brow_features']  # [bl, br, b_avg, furrow]
                    feat = np.concatenate([eye, brow])  # (7,)
                    label = self._heuristic_label(eye[2], brow[2], brow[3])
                    X.append(feat)
                    y.append(label)
                except Exception:
                    continue
            print(f'Real ISL eye/brow samples: {len(X)}')

        # Synthetic augmentation
        for _ in range(n_synth // 3):
            # Neutral
            X.append(self._synth(ear=0.28, brow=0.06, furrow=0.15, noise=0.015))
            y.append(0)
            # Raised brows (wh-question)
            X.append(self._synth(ear=0.34, brow=0.09, furrow=0.18, noise=0.015))
            y.append(1)
            # Furrowed brows (negation)
            X.append(self._synth(ear=0.22, brow=0.03, furrow=0.09, noise=0.015))
            y.append(2)

        X = np.array(X, dtype=np.float32)
        y = np.array(y, dtype=np.int64)
        perm = np.random.permutation(len(X))
        return torch.tensor(X[perm]), torch.tensor(y[perm])

    @staticmethod
    def _heuristic_label(ear_avg, brow_raise, furrow):
        if ear_avg > 0.30 and brow_raise > 0.07:
            return 1   # raised
        elif ear_avg < 0.25 or furrow < 0.11:
            return 2   # furrowed
        return 0       # neutral

    @staticmethod
    def _synth(ear, brow, furrow, noise):
        n = np.random.randn(7) * noise
        return np.array([
            ear+n[0], ear+n[1], ear+n[2],
            brow+n[3], brow+n[4],
            brow+n[5], furrow+n[6]
        ], dtype=np.float32)

    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


print('Dataset classes defined.')

In [ ]:
# ─── Instantiate DataLoaders ──────────────────────────────────────────────────
# Set max_samples to None for full AffectNet (440K images)
# Use e.g. 50000 for faster dev iteration
MAX_AFFECTNET = None   # ← change to int for quick testing

expr_train_ds = AffectNetDataset(AFFECTNET, 'train', train_tf, MAX_AFFECTNET)
expr_val_ds   = AffectNetDataset(AFFECTNET, 'val',   val_tf,   None)

# Balanced sampling for AffectNet (handles class imbalance)
label_counts  = Counter(expr_train_ds.labels)
weights = [1.0 / label_counts[int(l)] for l in expr_train_ds.labels]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

expr_train_dl = DataLoader(expr_train_ds, batch_size=128, sampler=sampler,
                            num_workers=4, pin_memory=True)
expr_val_dl   = DataLoader(expr_val_ds,   batch_size=128, shuffle=False,
                            num_workers=4, pin_memory=True)

# Head movement
split = int(0.8 * len(X_hm))
hm_train_ds = HeadMovementDataset(X_hm[:split], y_hm[:split], augment=True)
hm_val_ds   = HeadMovementDataset(X_hm[split:], y_hm[split:], augment=False)
hm_train_dl = DataLoader(hm_train_ds, batch_size=256, shuffle=True, num_workers=2)
hm_val_dl   = DataLoader(hm_val_ds,   batch_size=256, shuffle=False, num_workers=2)

# Eye/Brow
eb_ds = EyeBrowDataset(df_lm=df_isl_lm, n_synthetic=6000)
eb_split = int(0.8 * len(eb_ds))
eb_train_ds, eb_val_ds = torch.utils.data.random_split(
    eb_ds, [eb_split, len(eb_ds) - eb_split],
    generator=torch.Generator().manual_seed(SEED)
)
eb_train_dl = DataLoader(eb_train_ds, batch_size=512, shuffle=True)
eb_val_dl   = DataLoader(eb_val_ds,   batch_size=512, shuffle=False)

print(f'\nExpression  — Train: {len(expr_train_ds):,} | Val: {len(expr_val_ds):,}')
print(f'Head Mvmt   — Train: {len(hm_train_ds):,}   | Val: {len(hm_val_ds):,}')
print(f'Eye/Brow    — Train: {len(eb_train_ds):,}   | Val: {len(eb_val_ds):,}')

## 8. Model Architectures

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Stream 1: Expression CNN  (EfficientNet-B0 pretrained backbone)
# ═══════════════════════════════════════════════════════════════
class ExpressionCNN(nn.Module):
    def __init__(self, num_classes: int = 8, feat_dim: int = 256):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        bb_dim = self.backbone.num_features  # 1280
        self.head = nn.Sequential(
            nn.Linear(bb_dim, feat_dim), nn.BatchNorm1d(feat_dim), nn.GELU(), nn.Dropout(0.35)
        )
        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        feat  = self.backbone(x)       # (B, 1280)
        embed = self.head(feat)         # (B, feat_dim)
        return self.classifier(embed), embed


# ═══════════════════════════════════════════════════════════════
# Stream 2: Head Movement TCN
# ═══════════════════════════════════════════════════════════════
class CausalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, dilation=1, drop=0.2):
        super().__init__()
        pad = (kernel - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel, dilation=dilation, padding=pad)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, dilation=dilation, padding=pad)
        self.norm1 = nn.BatchNorm1d(out_ch)
        self.norm2 = nn.BatchNorm1d(out_ch)
        self.drop  = nn.Dropout(drop)
        self.act   = nn.GELU()
        self.res   = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        T = x.size(-1)
        o = self.act(self.norm1(self.conv1(x)[..., :T]))
        o = self.drop(self.act(self.norm2(self.conv2(o)[..., :T])))
        return o + self.res(x)


class HeadMovementTCN(nn.Module):
    def __init__(self, in_ch: int = 3, num_classes: int = 5,
                  channels=(64, 128, 256), feat_dim: int = 128):
        super().__init__()
        blocks, prev = [], in_ch
        for i, ch in enumerate(channels):
            blocks.append(CausalBlock(prev, ch, dilation=2**i))
            prev = ch
        self.tcn  = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(prev, feat_dim), nn.LayerNorm(feat_dim), nn.GELU(), nn.Dropout(0.2)
        )
        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, x):          # x: (B, T, 3)
        x = x.transpose(1, 2)      # (B, 3, T)
        f = self.pool(self.tcn(x)).squeeze(-1)  # (B, C)
        e = self.head(f)
        return self.classifier(e), e


# ═══════════════════════════════════════════════════════════════
# Stream 3: Eye/Brow MLP
# ═══════════════════════════════════════════════════════════════
class EyeBrowMLP(nn.Module):
    def __init__(self, in_dim: int = 7, num_classes: int = 3, feat_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, feat_dim), nn.LayerNorm(feat_dim), nn.GELU(), nn.Dropout(0.1),
        )
        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        e = self.net(x)
        return self.classifier(e), e


# ═══════════════════════════════════════════════════════════════
# Fusion: Late-fusion NMF classifier
# ═══════════════════════════════════════════════════════════════
class NMFFusion(nn.Module):
    """
    Combines embeddings from all three streams.
    Output: ISL NMF category
    """
    NMF_CLASSES = ['neutral', 'raised_brows', 'furrowed_brows', 'nod', 'shake', 'tilt', 'positive']

    def __init__(self, expr_dim=256, hm_dim=128, eb_dim=64):
        super().__init__()
        d = expr_dim + hm_dim + eb_dim
        self.net = nn.Sequential(
            nn.Linear(d, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, len(self.NMF_CLASSES))
        )

    def forward(self, expr_e, hm_e, eb_e):
        return self.net(torch.cat([expr_e, hm_e, eb_e], dim=-1))


# Instantiate
model_expr = ExpressionCNN(num_classes=NUM_EXPR_CLASSES, feat_dim=256).to(DEVICE)
model_hm   = HeadMovementTCN(num_classes=len(HEAD_CLASSES), feat_dim=128).to(DEVICE)
model_eb   = EyeBrowMLP(num_classes=3, feat_dim=64).to(DEVICE)
model_fuse = NMFFusion(expr_dim=256, hm_dim=128, eb_dim=64).to(DEVICE)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

for name, m in [('ExpressionCNN', model_expr), ('HeadMovementTCN', model_hm),
                 ('EyeBrowMLP', model_eb), ('NMFFusion', model_fuse)]:
    print(f'{name:<18}: {count_params(m):>10,} parameters')

## 9. Training Utilities

In [ ]:
class Meter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = self.avg = self.sum = self.count = 0
    def update(self, v, n=1):
        self.sum += v * n; self.count += n
        self.avg = self.sum / self.count

def top1_acc(logits, targets):
    return (logits.argmax(1) == targets).float().mean().item() * 100


def train_one_epoch(model, loader, opt, crit, device, name):
    model.train()
    lm, am = Meter(), Meter()
    for x, y in tqdm(loader, desc=f'  [{name}] train', leave=False):
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits, _ = model(x)
        loss = crit(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        lm.update(loss.item(), x.size(0))
        am.update(top1_acc(logits, y), x.size(0))
    return lm.avg, am.avg


@torch.no_grad()
def validate(model, loader, crit, device, name):
    model.eval()
    lm, am = Meter(), Meter()
    all_p, all_y = [], []
    for x, y in tqdm(loader, desc=f'  [{name}] val  ', leave=False):
        x, y = x.to(device), y.to(device)
        logits, _ = model(x)
        lm.update(crit(logits, y).item(), x.size(0))
        am.update(top1_acc(logits, y), x.size(0))
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_y.extend(y.cpu().numpy())
    return lm.avg, am.avg, np.array(all_p), np.array(all_y)


def plot_training(hist, title, save_path=None):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.plot(hist['tl'], label='Train', color='#E85D24')
    a1.plot(hist['vl'], label='Val',   color='#3B8BD4')
    a1.set(title='Loss', xlabel='Epoch'); a1.legend()
    a2.plot(hist['ta'], label='Train', color='#E85D24')
    a2.plot(hist['va'], label='Val',   color='#3B8BD4')
    a2.set(title='Accuracy (%)', xlabel='Epoch'); a2.legend()
    plt.suptitle(title); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=120)
    plt.show()


def save_ckpt(model, opt, epoch, val_acc, path):
    torch.save({'epoch': epoch, 'model': model.state_dict(),
                'opt': opt.state_dict(), 'val_acc': val_acc}, path)


def load_ckpt(model, path):
    if Path(path).exists():
        ck = torch.load(path, map_location=DEVICE)
        model.load_state_dict(ck['model'])
        print(f'Loaded {Path(path).name}  (val_acc={ck["val_acc"]:.2f}%)')


print('Training utilities ready.')

## 10. Train Stream 1 — Expression CNN on AffectNet

In [ ]:
EXPR_EPOCHS = 25

# Freeze backbone for first 5 epochs, then unfreeze
for p in model_expr.backbone.parameters():
    p.requires_grad = False

crit_expr = nn.CrossEntropyLoss(label_smoothing=0.1)
opt_expr  = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_expr.parameters()),
                               lr=3e-4, weight_decay=1e-4)
sched_expr = torch.optim.lr_scheduler.CosineAnnealingLR(opt_expr, EXPR_EPOCHS, eta_min=1e-6)

hist_expr = {'tl': [], 'vl': [], 'ta': [], 'va': []}
best_expr = 0

print('=== Training Expression CNN (AffectNet) ===')
for ep in range(1, EXPR_EPOCHS + 1):

    # Unfreeze backbone at epoch 6
    if ep == 6:
        print('  → Unfreezing EfficientNet backbone')
        for p in model_expr.backbone.parameters():
            p.requires_grad = True
        opt_expr = torch.optim.AdamW(model_expr.parameters(), lr=5e-5, weight_decay=1e-4)
        sched_expr = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt_expr, EXPR_EPOCHS - ep + 1, eta_min=1e-7)

    tl, ta = train_one_epoch(model_expr, expr_train_dl, opt_expr, crit_expr, DEVICE, 'Expr')
    vl, va, _, _ = validate(model_expr, expr_val_dl, crit_expr, DEVICE, 'Expr')
    sched_expr.step()

    for k, v in zip(['tl','vl','ta','va'], [tl, vl, ta, va]):
        hist_expr[k].append(v)

    if va > best_expr:
        best_expr = va
        save_ckpt(model_expr, opt_expr, ep, va, CKPT / 'expr_best.pt')

    print(f'Ep {ep:02d}/{EXPR_EPOCHS} | '
          f'Train loss={tl:.4f} acc={ta:.1f}% | '
          f'Val loss={vl:.4f} acc={va:.1f}% | Best={best_expr:.1f}%')

plot_training(hist_expr, 'Expression CNN — AffectNet', LOGS / 'expr_train.png')
print(f'\nBest val accuracy: {best_expr:.2f}%')

## 11. Fine-tune Expression CNN on ISL-CSLTR Frames

In [ ]:
# ─── Fine-tune expression model on ISL face crops ────────────────────────────
# ISL frames have real signer faces — fine-tuning improves domain adaptation

class ISLFaceCropDataset(Dataset):
    """
    Crops face regions from ISL-CSLTR frames using saved bbox.
    Labels are pseudo-labels from expression model inference.
    """
    def __init__(self, df_lm: pd.DataFrame, transform=None, pseudo_model=None, device='cpu'):
        self.transform = transform
        self.items = []

        if df_lm.empty:
            print('No ISL frames — skipping ISL face crop dataset.')
            return

        if pseudo_model is not None:
            print('Generating pseudo-labels from expression model...')
            pseudo_model.eval()

        for _, row in tqdm(df_lm.iterrows(), total=len(df_lm),
                           desc='Loading face crops'):
            try:
                npz = np.load(row['lm_path'])
                x1, y1, x2, y2 = npz['face_bbox'].astype(int)
            except Exception:
                continue

            img = cv2.imread(row['frame_path'])
            if img is None or x2 <= x1 or y2 <= y1:
                continue

            face = img[y1:y2, x1:x2]
            if face.size == 0:
                continue

            # Pseudo-label
            if pseudo_model is not None:
                rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
                t = val_tf(image=rgb)['image'].unsqueeze(0).to(device)
                with torch.no_grad():
                    lbl = pseudo_model(t)[0].argmax(1).item()
            else:
                lbl = 0  # placeholder

            self.items.append((face, lbl))

        print(f'ISL face crops loaded: {len(self.items)}')

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        face, label = self.items[idx]
        rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        if self.transform:
            rgb = self.transform(image=rgb)['image']
        return rgb, label


# Load best expression checkpoint, generate pseudo-labels, fine-tune
load_ckpt(model_expr, CKPT / 'expr_best.pt')

isl_face_ds = ISLFaceCropDataset(df_isl_lm, train_tf, model_expr, DEVICE)

if len(isl_face_ds) > 50:
    isl_face_dl = DataLoader(isl_face_ds, batch_size=32, shuffle=True, num_workers=2)

    FT_EPOCHS = 10
    opt_ft    = torch.optim.AdamW(model_expr.parameters(), lr=1e-5, weight_decay=1e-4)
    sched_ft  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ft, FT_EPOCHS, eta_min=1e-8)

    print(f'\n=== Fine-tuning on {len(isl_face_ds)} ISL face crops ===')
    for ep in range(1, FT_EPOCHS + 1):
        tl, ta = train_one_epoch(model_expr, isl_face_dl, opt_ft, crit_expr, DEVICE, 'FT')
        sched_ft.step()
        print(f'  FT Ep {ep:02d}/{FT_EPOCHS} | loss={tl:.4f} acc={ta:.1f}%')

    save_ckpt(model_expr, opt_ft, FT_EPOCHS, ta, CKPT / 'expr_finetuned.pt')
    print('Fine-tuned model saved.')
else:
    print('Skipping fine-tune — insufficient ISL face crops.')
    print('Download full ISL-CSLTR and re-run frame extraction.')

## 12. Train Stream 2 — Head Movement TCN

In [ ]:
HM_EPOCHS = 40
crit_hm   = nn.CrossEntropyLoss()
opt_hm    = torch.optim.Adam(model_hm.parameters(), lr=1e-3)
sched_hm  = torch.optim.lr_scheduler.OneCycleLR(
    opt_hm, max_lr=1e-3,
    steps_per_epoch=len(hm_train_dl), epochs=HM_EPOCHS
)

hist_hm = {'tl': [], 'vl': [], 'ta': [], 'va': []}
best_hm = 0

print('=== Training Head Movement TCN (ISL-CSLTR + synthetic) ===')
for ep in range(1, HM_EPOCHS + 1):
    model_hm.train()
    lm, am = Meter(), Meter()
    for x, y in tqdm(hm_train_dl, desc=f'  [HM] train ep{ep}', leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt_hm.zero_grad()
        logits, _ = model_hm(x)
        loss = crit_hm(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model_hm.parameters(), 1.0)
        opt_hm.step(); sched_hm.step()
        lm.update(loss.item(), x.size(0))
        am.update(top1_acc(logits, y), x.size(0))

    vl, va, _, _ = validate(model_hm, hm_val_dl, crit_hm, DEVICE, 'HM')
    for k, v in zip(['tl','vl','ta','va'], [lm.avg, vl, am.avg, va]):
        hist_hm[k].append(v)

    if va > best_hm:
        best_hm = va
        save_ckpt(model_hm, opt_hm, ep, va, CKPT / 'hm_best.pt')

    if ep % 5 == 0:
        print(f'Ep {ep:02d}/{HM_EPOCHS} | '
              f'Train loss={lm.avg:.4f} acc={am.avg:.1f}% | '
              f'Val loss={vl:.4f} acc={va:.1f}% | Best={best_hm:.1f}%')

plot_training(hist_hm, 'Head Movement TCN', LOGS / 'hm_train.png')
print(f'\nBest val accuracy: {best_hm:.2f}%')

## 13. Train Stream 3 — Eye/Brow MLP

In [ ]:
EB_EPOCHS = 30
crit_eb   = nn.CrossEntropyLoss()
opt_eb    = torch.optim.AdamW(model_eb.parameters(), lr=5e-4, weight_decay=1e-3)
sched_eb  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_eb, EB_EPOCHS, eta_min=1e-6)

hist_eb = {'tl': [], 'vl': [], 'ta': [], 'va': []}
best_eb = 0

print('=== Training Eye/Brow MLP (ISL-CSLTR landmarks + synthetic) ===')
for ep in range(1, EB_EPOCHS + 1):
    tl, ta = train_one_epoch(model_eb, eb_train_dl, opt_eb, crit_eb, DEVICE, 'EB')
    vl, va, _, _ = validate(model_eb, eb_val_dl, crit_eb, DEVICE, 'EB')
    sched_eb.step()
    for k, v in zip(['tl','vl','ta','va'], [tl, vl, ta, va]):
        hist_eb[k].append(v)
    if va > best_eb:
        best_eb = va
        save_ckpt(model_eb, opt_eb, ep, va, CKPT / 'eb_best.pt')
    if ep % 5 == 0:
        print(f'Ep {ep:02d}/{EB_EPOCHS} | '
              f'Train loss={tl:.4f} acc={ta:.1f}% | '
              f'Val loss={vl:.4f} acc={va:.1f}% | Best={best_eb:.1f}%')

plot_training(hist_eb, 'Eye/Brow MLP', LOGS / 'eb_train.png')
print(f'\nBest val accuracy: {best_eb:.2f}%')

## 14. Evaluation & Confusion Matrices

In [ ]:
def plot_cm(preds, labels, class_names, title, save_path=None):
    cm = confusion_matrix(labels, preds)
    cm_n = cm.astype(float) / (cm.sum(1, keepdims=True) + 1e-8)
    fig, ax = plt.subplots(figsize=(max(6, len(class_names)), max(5, len(class_names)-1)))
    sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.4, ax=ax)
    ax.set(ylabel='True', xlabel='Predicted', title=title)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=120)
    plt.show()
    print(classification_report(labels, preds, target_names=class_names,
                                 zero_division=0))


# Load best checkpoints
load_ckpt(model_expr, CKPT / 'expr_best.pt')
load_ckpt(model_hm,   CKPT / 'hm_best.pt')
load_ckpt(model_eb,   CKPT / 'eb_best.pt')

# Evaluate
_, _, p_expr, y_expr = validate(model_expr, expr_val_dl,   crit_expr, DEVICE, 'Expr')
_, _, p_hm,   y_hm2  = validate(model_hm,   hm_val_dl,    crit_hm,   DEVICE, 'HM')
_, _, p_eb,   y_eb   = validate(model_eb,   eb_val_dl,    crit_eb,   DEVICE, 'EB')

print('\n━━━ Expression CNN (AffectNet → ISL) ━━━')
plot_cm(p_expr, y_expr, AFFECTNET_CLASS_NAMES[:NUM_EXPR_CLASSES],
         'Expression CNN', LOGS / 'cm_expr.png')

print('\n━━━ Head Movement TCN ━━━')
plot_cm(p_hm, y_hm2, list(HEAD_CLASSES.values()),
         'Head Movement TCN', LOGS / 'cm_hm.png')

print('\n━━━ Eye/Brow MLP ━━━')
plot_cm(p_eb, y_eb, list(EyeBrowDataset.CLASS_NAMES.values()),
         'Eye/Brow MLP', LOGS / 'cm_eb.png')

## 15. Ablation Study

In [ ]:
@torch.no_grad()
def stream_acc(model, loader, device):
    model.eval()
    c = t = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        c += (model(x)[0].argmax(1) == y).sum().item(); t += y.size(0)
    return 100 * c / t

results = {
    'Expression  (8-class, AffectNet)':  stream_acc(model_expr, expr_val_dl, DEVICE),
    'Head mvmt   (5-class, ISL+synth)':  stream_acc(model_hm,   hm_val_dl,   DEVICE),
    'Eye/brow    (3-class, ISL+synth)':  stream_acc(model_eb,   eb_val_dl,   DEVICE),
    'Random baseline (expr, 8-class)' :  100 / NUM_EXPR_CLASSES,
    'Random baseline (HM, 5-class)'   :  100 / 5,
    'Random baseline (EB, 3-class)'   :  100 / 3,
}

print('\n=== Ablation Results ===')
for name, acc in results.items():
    bar = '█' * int(acc / 2)
    print(f'  {name:<45} {acc:5.1f}%  {bar}')

fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#3B8BD4','#3B8BD4','#3B8BD4','#BDBDBD','#BDBDBD','#BDBDBD']
names  = list(results.keys())
accs   = list(results.values())
bars   = ax.barh(names, accs, color=colors, edgecolor='none', height=0.55)
for b, a in zip(bars, accs):
    ax.text(b.get_width() + 0.5, b.get_y() + b.get_height()/2,
             f'{a:.1f}%', va='center', fontsize=10)
ax.set(xlabel='Accuracy (%)', xlim=(0, 112),
        title='Ablation Study — Per-stream accuracy vs random baseline')
plt.tight_layout()
plt.savefig(LOGS / 'ablation.png', dpi=120)
plt.show()

## 16. Real-time Inference Pipeline

In [ ]:
class ISLNMFPipeline:
    """
    End-to-end NMF detection on live video or single frames.
    Maintains a rolling buffer of head poses for TCN inference.
    """
    EXPR_NAMES = AFFECTNET_CLASS_NAMES
    HM_NAMES   = list(HEAD_CLASSES.values())
    EB_NAMES   = list(EyeBrowDataset.CLASS_NAMES.values())

    # Map expression to ISL grammar function
    GRAMMAR = {
        'neutral':   'Statement / no NMF',
        'happy':     'Affirmative marker',
        'surprise':  'Wh-question (who/what/where)',
        'fear':      'Wh-question (secondary)',
        'sad':       'Negation / emotional tone',
        'disgust':   'Negation',
        'anger':     'Negation / intensity',
        'contempt':  'Negation',
        'nod':       'Yes / affirmative / sentence end',
        'shake':     'No / negation',
        'tilt':      'Conditional / topic marker',
        'forward':   'Emphasis / assertion',
        'raised_brows': 'Wh-question facial marker',
        'furrowed_brows': 'Negation / yes-no question',
    }

    def __init__(self, expr_model, hm_model, eb_model,
                  device, seq_len=SEQ_LEN):
        self.expr  = expr_model.eval()
        self.hm    = hm_model.eval()
        self.eb    = eb_model.eval()
        self.dev   = device
        self.seq_len = seq_len
        self.pose_buf = []
        self.face_mesh = mp_face.FaceMesh(
            static_image_mode=False, max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

    @torch.no_grad()
    def process(self, frame_bgr: np.ndarray) -> dict:
        out = {'face': False, 'expression': None,
               'head_movement': None, 'eye_brow': None, 'grammar': []}

        feat = extract_features(frame_bgr, self.face_mesh)
        if feat is None:
            return out
        out['face'] = True

        # ── Head movement ──
        self.pose_buf.append(feat['head_pose'])
        if len(self.pose_buf) > self.seq_len:
            self.pose_buf.pop(0)
        if len(self.pose_buf) == self.seq_len:
            seq = np.stack(self.pose_buf)
            seq = (seq - seq.mean(0)) / (seq.std(0) + 1e-6)
            t = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(self.dev)
            probs = F.softmax(self.hm(t)[0], 1)[0]
            cls   = probs.argmax().item()
            out['head_movement'] = {'label': self.HM_NAMES[cls],
                                     'conf':  probs[cls].item()}
            out['grammar'].append(self.GRAMMAR.get(self.HM_NAMES[cls], ''))

        # ── Eye/brow ──
        eb_v = torch.tensor(
            np.concatenate([feat['eye_features'], feat['brow_features']]),
            dtype=torch.float32
        ).unsqueeze(0).to(self.dev)
        probs_eb = F.softmax(self.eb(eb_v)[0], 1)[0]
        cls_eb   = probs_eb.argmax().item()
        out['eye_brow'] = {'label': self.EB_NAMES[cls_eb],
                            'conf':  probs_eb[cls_eb].item()}
        out['grammar'].append(self.GRAMMAR.get(self.EB_NAMES[cls_eb], ''))

        # ── Expression (face crop) ──
        x1, y1, x2, y2 = feat['face_bbox']
        if x2 > x1 and y2 > y1:
            crop = frame_bgr[y1:y2, x1:x2]
            rgb  = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            img_t = val_tf(image=rgb)['image'].unsqueeze(0).to(self.dev)
            probs_e = F.softmax(self.expr(img_t)[0], 1)[0]
            cls_e   = probs_e.argmax().item()
            out['expression'] = {'label': self.EXPR_NAMES[cls_e],
                                  'conf':  probs_e[cls_e].item()}
            out['grammar'].append(self.GRAMMAR.get(self.EXPR_NAMES[cls_e], ''))

        return out

    def annotate(self, frame_bgr: np.ndarray, result: dict) -> np.ndarray:
        out = frame_bgr.copy()
        if not result['face']:
            cv2.putText(out, 'No face', (10, 30),
                         cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,200), 2)
            return out
        y0 = 28
        for key, col in [('expression', (0,180,60)),
                          ('head_movement', (50,120,255)),
                          ('eye_brow', (200,100,0))]:
            pred = result.get(key)
            if pred:
                text = f"{key}: {pred['label']} ({pred['conf']:.0%})"
                cv2.putText(out, text, (10, y0),
                             cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)
                y0 += 26
        # Grammar interpretation
        grammar = list(dict.fromkeys([g for g in result['grammar'] if g]))
        if grammar:
            cv2.putText(out, f"ISL grammar: {' | '.join(grammar[:2])}",
                         (10, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (220,220,220), 1)
        return out


pipeline = ISLNMFPipeline(model_expr, model_hm, model_eb, DEVICE)
print('Pipeline ready.')
print("""
── Webcam demo ────────────────────────────────────────
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret: break
    result   = pipeline.process(frame)
    annotated = pipeline.annotate(frame, result)
    cv2.imshow('ISL NMF', annotated)
    if cv2.waitKey(1) & 0xFF == ord('q'): break
cap.release(); cv2.destroyAllWindows()
───────────────────────────────────────────────────────
""")

## 17. Test on ISL-CSLTR Frame Sample

In [ ]:
# Test inference on a sequence of ISL video frames

def run_inference_on_video_segment(video_id: str, df_frames: pd.DataFrame,
                                    pipeline, n_frames: int = 60):
    """Run NMF inference on the first n_frames of a given video."""
    if df_frames.empty:
        print('No frames available.'); return

    vids = df_frames['video_id'].unique()
    if video_id not in vids:
        video_id = vids[0]
        print(f'video_id not found, using: {video_id}')

    frames = df_frames[df_frames['video_id'] == video_id]\
             .sort_values('frame_path')['frame_path'].tolist()[:n_frames]

    results = []
    for fp in tqdm(frames, desc='Inference'):
        img = cv2.imread(fp)
        if img is None: continue
        results.append(pipeline.process(img))

    # Summarise
    print(f'\n── NMF Summary for {video_id} ({len(results)} frames) ──')
    for key in ['expression', 'head_movement', 'eye_brow']:
        labels = [r[key]['label'] for r in results if r[key]]
        if labels:
            most_common = Counter(labels).most_common(3)
            print(f'  {key:<15}: {most_common}')

    # Plot annotated sample frames
    n_show = min(8, len(frames))
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    for i, (fp, res) in enumerate(zip(frames[:n_show], results[:n_show])):
        ax = axes[i // 4][i % 4]
        img = cv2.imread(fp)
        annotated = pipeline.annotate(img, res)
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.axis('off')
    plt.suptitle(f'ISL NMF Inference — {video_id}', fontsize=12)
    plt.tight_layout()
    plt.savefig(LOGS / 'inference_sample.png', dpi=100)
    plt.show()


if not df_isl_frames.empty:
    first_vid = df_isl_frames['video_id'].iloc[0]
    run_inference_on_video_segment(first_vid, df_isl_frames, pipeline, n_frames=60)
else:
    print('Run frame extraction (Section 4) first.')

## 18. ONNX Export

In [ ]:
ONNX_DIR = BASE / 'onnx'
ONNX_DIR.mkdir(exist_ok=True)

class LogitsWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x)[0]

exports = [
    (model_expr, torch.randn(1, 3, 112, 112).to(DEVICE),
     'expression_cnn.onnx', ['face_image'], ['expr_logits'],
     {'face_image': {0: 'B'}, 'expr_logits': {0: 'B'}}),

    (model_hm,   torch.randn(1, SEQ_LEN, 3).to(DEVICE),
     'head_movement_tcn.onnx', ['pose_seq'], ['hm_logits'],
     {'pose_seq': {0: 'B'}, 'hm_logits': {0: 'B'}}),

    (model_eb,   torch.randn(1, 7).to(DEVICE),
     'eyebrow_mlp.onnx', ['eb_feats'], ['eb_logits'],
     {'eb_feats': {0: 'B'}, 'eb_logits': {0: 'B'}}),
]

for model, dummy, fname, inp, outp, dax in exports:
    model.eval()
    wrapper = LogitsWrapper(model)
    path = ONNX_DIR / fname
    torch.onnx.export(
        wrapper, dummy, path,
        export_params=True, opset_version=17,
        input_names=inp, output_names=outp,
        dynamic_axes=dax
    )
    size_mb = path.stat().st_size / 1e6
    print(f'  {fname:<30}  {size_mb:.1f} MB')

print('\nAll models exported.')
print('Inference with ONNX Runtime:')
print("""
  import onnxruntime as ort
  sess = ort.InferenceSession('onnx/head_movement_tcn.onnx',
                               providers=['CUDAExecutionProvider','CPUExecutionProvider'])
  logits = sess.run(['hm_logits'], {'pose_seq': seq_np})[0]
""")

## 19. What's Next

### Using full ISL-CSLTR (all 700 videos)
Change `max_videos=50` → `max_videos=None` in Section 4 and re-run.

### Manual NMF annotation (recommended for production)
Use **ELAN** (https://archive.mpi.nl/tla/elan) to annotate NMF tiers on the 700 videos:
- Tier 1: `head_movement` → nod / shake / tilt / still  
- Tier 2: `brow` → raised / furrowed / neutral  
- Tier 3: `expression` → map to ISL grammar category  
Then replace the heuristic labels with real ELAN annotations.

### Cross-attention fusion (next-level)
Replace `NMFFusion` with a cross-attention transformer:
```python
# Pseudo-code
face_tok = face_encoder(face_seq)   # (B, Tf, D)
hand_tok = hand_encoder(hand_seq)   # (B, Th, D)
fused = cross_attention(face_tok, hand_tok)
nmf_label = classifier(fused.mean(1))
```

### Add hand keypoints (complete ISL system)
Use MediaPipe Hands alongside Face Mesh:
```python
mp_hands = mp.solutions.hands
# Run in same frame → get 21 hand landmarks per hand
# Feed to a separate hand stream
# Fuse with face NMF streams
```

### Signer-independent evaluation
Split ISL-CSLTR by signer ID (7 signers), use leave-one-signer-out CV
to get realistic performance estimates.